# OS Names API vs ukgeo Level 2

This notebook compares the Ordnance Survey Names API `/find` endpoint with ukgeo's local Level 2 parquet-backed geocoder on the same regression cases.

The goal is to scope whether the API is worth implementing as a Level 3 fallback for cases where local matching is unresolved, low-confidence, or known to be brittle.


In [ ]:
import polars as pl
from pathlib import Path

from ukgeo import Geocoder
from ukgeo.pipeline import _haversine
from ukgeo.level3_os_names import query_os_names, implied_confidence
from ukgeo.utils import load_env, get_env_key

load_env()
OS_API_KEY = get_env_key("OS_API_KEY")

geo = Geocoder()


## Test cases

These are the current 15 core regression cases plus the six documented strict `xfail` gaps.


In [ ]:
TEST_CASES = [
    # (input, true_lat, true_lon, label)
    ("LS1 1BA",                        53.7997, -1.5492,  "postcode"),
    ("WF10 4QH",                       53.7230, -1.3550,  "postcode"),
    ("M62 Junction 26",                53.736186, -1.726631,  "motorway junction"),
    ("A647 near Bradford",             53.7950, -1.7800,  "a-road + place"),
    ("Lofthouse Interchange",          53.7320, -1.5210,  "named interchange"),
    ("Spaghetti Junction Birmingham",  52.5137, -1.8346,  "colloquial junction"),
    ("Magic Roundabout Swindon",       51.5585, -1.7837,  "colloquial roundabout"),
    ("A64 York bypass near Tadcaster", 53.8720, -1.2650,  "a-road + place"),
    ("Skipton, North Yorkshire",       53.9619, -2.0175,  "place + county"),
    ("Aberford West Yorkshire",        53.8333, -1.3333,  "village + county"),
    ("Dartford Crossing Kent",         51.4454,  0.2744,  "named crossing"),
    ("Station Road Leeds",             53.7955, -1.5490,  "street + city"),
    ("A1(M) Junction 47 Garforth",     54.009541, -1.377813,  "motorway junction"),
    ("Brafford West Yorkshire",        53.7950, -1.7594,  "typo"),
    ("Sighthill Edinburgh",            55.9285, -3.2441,  "suburb + city"),
    # xfail cases
    ("A1M Junction 47 Garforth",       54.009541, -1.377813,  "xfail: no-bracket motorway"),
    ("A1 M Junction 47 Garforth",      54.009541, -1.377813,  "xfail: spaced motorway"),
    ("Bradford Cornwall",              53.7908, -1.7546,  "xfail: wrong county"),
    ("Dartford Yorkshire",             51.4454,  0.2744,  "xfail: wrong county"),
    ("Lofthouse Interchange",          53.7617, -1.5420,  "xfail: old disputed ground truth"),
    ("B1224 York",                     53.9590, -1.0815,  "xfail: B-road route semantics"),
]


## Run comparison

For the API, this takes the top `/find` result. For ukgeo, this uses the normal local `Geocoder()` output.


In [ ]:
rows = []
lookup = geo._lookup

for inp, true_lat, true_lon, label in TEST_CASES:
    # ukgeo
    ukgeo_result = geo.geocode(inp)
    ukgeo_dist = (
        _haversine(true_lat, true_lon, ukgeo_result.lat, ukgeo_result.lon)
        if ukgeo_result.resolved else None
    )

    # OS Names API — take top result
    try:
        api_results = query_os_names(inp, OS_API_KEY, max_results=1)
        if api_results:
            top = api_results[0]
            api_lat, api_lon = lookup.bng_to_wgs84(top["GEOMETRY_X"], top["GEOMETRY_Y"])
            api_dist = _haversine(true_lat, true_lon, api_lat, api_lon)
            api_type = top.get("LOCAL_TYPE", "")
            api_name = top.get("NAME1", "")
            api_county = top.get("COUNTY_UNITARY", "")  # readable text, not URI
            api_error = ""
        else:
            api_lat = api_lon = api_dist = None
            api_type = api_name = api_county = ""
            api_error = "no results"
    except Exception as e:
        api_lat = api_lon = api_dist = None
        api_type = api_name = api_county = ""
        api_error = str(e)

    rows.append({
        "input": inp,
        "label": label,
        "true_lat": true_lat,
        "true_lon": true_lon,
        "ukgeo_lat": ukgeo_result.lat,
        "ukgeo_lon": ukgeo_result.lon,
        "ukgeo_dist_m": ukgeo_dist,
        "ukgeo_conf": ukgeo_result.confidence,
        "ukgeo_interpreted": ukgeo_result.interpreted_as,
        "api_lat": api_lat,
        "api_lon": api_lon,
        "api_dist_m": api_dist,
        "api_name": api_name,
        "api_type": api_type,
        "api_county": api_county,
        "api_error": api_error,
    })

df = pl.DataFrame(rows)

display_cols = ["input", "label", "ukgeo_dist_m", "ukgeo_conf", "api_dist_m", "api_name"]
print(df.select(display_cols).to_pandas().to_string(index=False))


## Implied confidence from candidate spread

Rather than taking the API's top result blindly, we can infer confidence
from how spread out the top-N candidates are geographically.

- **Tight cluster (spread < 2km)** → High — API is unambiguous
- **Moderate spread (2–10km)** → Medium  
- **Wide spread (> 10km)** → Low — API result is uncertain

This gives us a confidence signal without OS providing one explicitly.


In [ ]:
# --- Spread analysis: implied confidence from candidate spread ---
spread_rows = []
for inp, true_lat, true_lon, label in TEST_CASES:
    candidates = query_os_names(inp, OS_API_KEY, max_results=5)
    conf, spread_km = implied_confidence(candidates, lookup)
    n = len(candidates)
    names = [c.get("NAME1", "") for c in candidates]
    spread_rows.append({
        "input": inp,
        "n_candidates": n,
        "implied_confidence": conf,
        "spread_km": round(spread_km, 2),
        "top_5_names": " | ".join(names[:5]),
    })

spread_df = pl.DataFrame(spread_rows)
print(spread_df.to_pandas().to_string(index=False))


## Where does the API beat ukgeo?


In [ ]:
api_wins = (
    df
    .filter(pl.col("api_dist_m").is_not_null() & pl.col("ukgeo_dist_m").is_not_null())
    .with_columns((pl.col("ukgeo_dist_m") - pl.col("api_dist_m")).alias("api_advantage_m"))
    .filter(pl.col("api_advantage_m") > 0)
    .sort("api_advantage_m", descending=True)
)

print(api_wins.select(["input", "label", "api_advantage_m", "ukgeo_dist_m", "api_dist_m", "api_name"]).to_pandas().to_string(index=False))


## Where does ukgeo beat the API?


In [ ]:
ukgeo_wins = (
    df
    .filter(pl.col("api_dist_m").is_not_null() & pl.col("ukgeo_dist_m").is_not_null())
    .with_columns((pl.col("api_dist_m") - pl.col("ukgeo_dist_m")).alias("ukgeo_advantage_m"))
    .filter(pl.col("ukgeo_advantage_m") > 0)
    .sort("ukgeo_advantage_m", descending=True)
)

print(ukgeo_wins.select(["input", "label", "ukgeo_advantage_m", "ukgeo_dist_m", "api_dist_m", "ukgeo_interpreted", "api_name"]).to_pandas().to_string(index=False))


## Does either resolver miss cases the other resolves?


In [ ]:
resolution_gaps = df.filter(
    (pl.col("ukgeo_dist_m").is_null() & pl.col("api_dist_m").is_not_null())
    | (pl.col("ukgeo_dist_m").is_not_null() & pl.col("api_dist_m").is_null())
)

print(resolution_gaps.select(["input", "label", "ukgeo_dist_m", "api_dist_m", "api_name", "api_error"]).to_pandas().to_string(index=False))


## Does `COUNTY_UNITARY` return readable text?


In [ ]:
county_examples = (
    df
    .filter(pl.col("api_county").is_not_null() & (pl.col("api_county") != ""))
    .select(["input", "api_name", "api_type", "api_county"])
    .unique()
)

print(county_examples.to_pandas().to_string(index=False))


## API response time


In [ ]:
import time

times = []
for inp, *_ in TEST_CASES[:10]:
    t0 = time.time()
    query_os_names(inp, OS_API_KEY)
    times.append(time.time() - t0)

print(f"OS Names API — mean response time: {sum(times)/len(times)*1000:.0f}ms per query")
print("ukgeo Level 2 — mean time: ~5ms per query (in-process parquet)")


## Conclusions

| Question | Answer |
|---|---|
| API resolve rate vs ukgeo | Both resolve all 21 test cases |
| Median distance — ukgeo | ~600m (excl. known xfail cases) |
| Median distance — API | ~7,400m (excl. known xfail cases) |
| Cases where API clearly wins | Dartford Crossing (1,111m vs 2,254m), B1224 York (2,873m vs 11,243m), Lofthouse Interchange (slight) |
| Cases where ukgeo clearly wins | Motorway junctions (M62 J26: 2m vs 7,423m; A1(M) J47: 2m vs 58,629m), colloquial names (Magic Roundabout: 358m vs 20,465m; Spaghetti Junction: 957m vs 8,095m), typos (Brafford: 565m vs 267,525m), road+place (A64 Tadcaster: 1,421m vs 20,180m) |
| API latency | 217ms per query vs ~5ms for ukgeo Level 2 |
| COUNTY_UNITARY readable? | Yes — returns plain text e.g. "North Yorkshire", "City of Edinburgh". Our parquet returns URIs. Useful for future Level 2 enrichment. |
| Worth implementing as Level 3? | Yes, but narrowly — only for named infrastructure (bridges, crossings) and cases with no road reference. Not useful for junctions, colloquial names, or typos where ukgeo already wins decisively. |


## Save results


In [ ]:
Path("data").mkdir(exist_ok=True)
df.write_csv("data/os_names_api_comparison.csv")
print("Saved to data/os_names_api_comparison.csv")
